0001167619 - Carmine Santella - carmine.santella@studio.unibo.it

# ML4CV Assignment
This is the main notebook for the ML4CV assignment. It contains the code for training and evaluating the model on the PRW dataset, as well as the report with the results and analysis.


In [ ]:
import numpy as np 
import pandas as pd
import os
import cv2
import scipy.io as sio
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
SCRIPTS_PATH = "/kaggle/input/datasets/gambesantella/ml4cv-scripts"   # contains all .py modules
DATA_ROOT    = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"  # PRW dataset
WEIGHTS_PATH = "/kaggle/input/models/gambesantella/final-model-ml4cv/pytorch/default/1/model_final_ML4CV.pt.pt"      # pretrained checkpoint

# PRW folder
IMG_DIR = os.path.join(DATA_ROOT, 'frames') 
ANNO_DIR = os.path.join(DATA_ROOT, 'annotations')

## 1. Dataset

The **PRW** (Person Re-identification in the Wild) dataset contains:
- **11,816** scene images captured in a university campus
- **932** labeled identities + distractors
- **2,057** query images

We use `PRWTrainDataset` for training, `PRWGalleryDataset` for gallery inference, and `PRWQueryDataset` for query embeddings.

The annotation format is `[x1, y1, w, h, pid]` in 1-indexed MATLAB coordinates, which we convert to 0-indexed `[x1, y1, x2, y2]`.

In [ ]:

# Verify paths exist before continuing
for label, p in [("scripts", SCRIPTS_PATH), ("PRW data", DATA_ROOT), ("weights", WEIGHTS_PATH)]:
    if not Path(p).exists():
        print(f"{label} not found at {p}")
    else:
        print(f"{label}: {p}")

# Make the scripts importable
if SCRIPTS_PATH not in sys.path:
    sys.path.insert(0, SCRIPTS_PATH)

In [ ]:
from prw import PRWTrainDataset, PRWGalleryDataset, PRWQueryDataset, collate_fn
from transforms import get_train_transforms, get_eval_transforms
from person_search_model import PersonSearchModel
from oim_loss import OIMLoss
from train import train_one_epoch, warmup_lr_scheduler
from evaluate import eval_search_prw, build_gallery_dets_feats, extract_query_feats

In [ ]:
# Load datasets
train_dataset = PRWTrainDataset(DATA_ROOT, transforms=get_train_transforms())
gallery_dataset = PRWGalleryDataset(DATA_ROOT, transforms=get_eval_transforms())
query_dataset = PRWQueryDataset(DATA_ROOT, transforms=get_eval_transforms())

NUM_PIDS = train_dataset.num_labeled_pids
print(f'Train frames: {len(train_dataset)}')
print(f'Gallery frames: {len(gallery_dataset)}')
print(f'Query images: {len(query_dataset)}')
print(f'Labeled identities (training): {NUM_PIDS}')

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset, batch_size=2, shuffle=True,
    num_workers=2, collate_fn=collate_fn, pin_memory=True
)
gallery_loader = DataLoader(
    gallery_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=collate_fn
)
query_loader = DataLoader(
    query_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=collate_fn
)

### Dataset visualisation

Let's inspect a training sample to confirm that bounding boxes are correctly aligned.

In [ ]:
import matplotlib.patches as patches
import torchvision.transforms.functional as TF

def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

img_t, target = train_dataset[0]
img_np = denormalize(img_t).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 1, figsize=(12, 7))
ax.imshow(img_np)
for box, pid in zip(target['boxes'], target['pids']):
    x1, y1, x2, y2 = box
    color = 'lime' if pid >= 0 else 'red'
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-4, f'pid={int(pid)}', color=color, fontsize=7)
ax.axis('off')
ax.set_title('Training sample (green=labeled, red=unlabeled)')
plt.tight_layout()
plt.show()

## 2. Model Architecture

### 2.1 Overview

Our model is an **end-to-end Person Search** network that jointly performs detection and Re-ID in a single forward pass. We build on a **Faster R-CNN with a ResNet-50 FPN v2 backbone** pre-trained on COCO, with two key modifications: the box predictor is replaced with a binary person/background head, and a dedicated **Re-ID branch with Norm-Aware Embedding (NAE)** is added next to it.

```
Image  →  [ResNet-50 + FPN v2]  →  feature pyramid
              ↓
     [Region Proposal Network]
              ↓
        proposals (RoIs)
              ↓
        [RoI Align]
              ↓
      roi_features  (N, 1024)
       /              \
[Box Predictor]    [Re-ID Head]
 cls + reg loss      ↓
                  (direction, norm)
                     │       │
                     │       └──► sigmoid → person score (inference)
                     │           BCE loss (training, supervision = fg/bg label)
                     │
                     └──► L2-normalized embedding (256-d)
                          OIM loss (training)
                          cosine similarity (inference)
```

### 2.2 Norm-Aware Embedding (NAE)

The Re-ID head produces a 256-d embedding through `Linear(1024→512) → BN → ReLU → Linear(512→256)`, then splits the result into two semantically distinct components:

- **Direction** (the L2-normalized embedding) encodes *identity* and is used for cosine similarity at the gallery–query matching stage.
- **Magnitude** (the embedding's L2 norm), passed through a learned `BatchNorm1d` and `sigmoid`, encodes *objectness* and replaces the FastRCNN classifier's softmax score at inference time.

The motivation is that re-identification ("which person?") and detection ("is this a person?") are fundamentally different questions. Forcing a shared embedding to encode both in its full dimensions creates a tension that compromises both. Decoupling them into orthogonal components of the same vector lets each branch optimize independently while still benefiting from a shared feature.

At inference, the norm-derived score is mathematically injected back into the classifier's logit position so torchvision's `postprocess_detections` (score thresholding, NMS, top-k) can be reused unchanged.

### 2.3 OIM Loss

The Online Instance Matching loss [Xiao et al., 2017] supervises the direction branch. It maintains two external memory banks:
- A **Lookup Table (LUT)**: one feature vector per labeled identity, updated via exponential moving average with momentum $m = 0.5$.
- A **Circular Queue (CQ)**: a FIFO buffer of size 5000 storing recent unlabeled person features, used as hard negatives.

The loss is a softmax cross-entropy over cosine similarities scaled by temperature $\tau = 30$:

$$\mathcal{L}_{\text{OIM}} = -\frac{1}{N} \sum_{i} \log \frac{e^{\tau \cdot \mathbf{f}_i^\top \mathbf{v}_{y_i}}}{\sum_{k} e^{\tau \cdot \mathbf{f}_i^\top \mathbf{v}_k} + \sum_{q} e^{\tau \cdot \mathbf{f}_i^\top \mathbf{u}_q}}$$

where $\mathbf{f}_i$ are labeled features, $\mathbf{v}_k$ are LUT entries, and $\mathbf{u}_q$ are CQ entries.

The NAE-norm BCE loss and the OIM direction loss are summed with the standard FastRCNN classification + box regression losses and back-propagated jointly.


In [ ]:
model = PersonSearchModel(
    num_pids       = NUM_PIDS,
    backbone_name  = "resnet50_v2",
    embed_dim      = 256,
    num_unlabeled  = 500,            # PRW: 500 (paper §Impl. Details); use 5000 for CUHK-SYSU
    oim_momentum   = 0.5,
    oim_scalar     = 30.0,
    triplet_weight = 0.0,
    nae_weight     = 1.0,
    pretrained     = True,
).to(device)

model.model.transform.min_size = (900,)
model.model.transform.max_size = 1500

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total  |  {trainable:,} trainable")
print(f"Transform: {model.model.transform.min_size}, max={model.model.transform.max_size}")

## 3. Training

### Training configuration

| Hyperparameter | Value |
|---|---|
| Optimizer | SGD (momentum=0.9, weight decay=5e-4, Nesterov) |
| LR (head) | 2e-3 |
| LR (backbone) | 2e-4 (10× lower than head) |
| LR schedule | **CosineAnnealingLR** over 26 epochs, $\eta_{\min} = 10^{-5}$ |
| Warm-up | Linear, 1000 iterations, factor 1/1000 |
| Total epochs | 26 |
| Batch size | 2 images |
| Input resolution | $\min = 900$, $\max = 1500$ |
| AMP | Enabled (`torch.cuda.amp`) |
| Gradient clipping | Max norm 10.0 |

We fine-tune from COCO pretrained weights with a 10× lower LR on the backbone parameters to preserve the pretrained features while adapting the new heads aggressively. ReID supervision is restricted to **foreground proposals only** so that the embedding head's BatchNorm sees a distribution consistent with what it encounters at inference (post-NMS detections, mostly foreground).


In [ ]:
import torch.nn as nn

# Freezing up to the 2nd convolutional stage 
FREEZE_LAYERS = (
    "body.conv1", 
    "body.bn1"
)

for name, param in model.model.backbone.named_parameters():
    if any(name.startswith(prefix) for prefix in FREEZE_LAYERS):
        param.requires_grad = False
 
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"After freezing: {trainable:,} trainable  |  {frozen:,} frozen")

### Training parameters
The `train` function accepts the following parameters:
- `num_epochs`: total number of training epochs (default: 26)
- `lr`: initial learning rate (default: 0.003)
- `batch_size`: number of images per batch (default: 2)
- `warmup_iters`: number of iterations for linear warm-up (default: 1000)
- `checkpoint_interval`: save a checkpoint every N epochs (default: 5)

In [ ]:
from torch.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
 
BASE_LR    = 2e-3
NUM_EPOCHS = 26
WARMUP_ITERS = 1000
 
backbone_params = [p for n, p in model.named_parameters()
                   if "backbone" in n and p.requires_grad]
head_params     = [p for n, p in model.named_parameters()
                   if "backbone" not in n and p.requires_grad]
 
optimizer = torch.optim.SGD(
    [
        {"params": backbone_params, "lr": BASE_LR / 10},
        {"params": head_params,     "lr": BASE_LR},
    ],
    momentum=0.9,     
    weight_decay=5e-4,
    nesterov=True,      # Nesterov momentum: slightly faster convergence
)
 
# Created ONCE here, passed into train_one_epoch every epoch
# (recreating GradScaler each epoch resets its internal loss scale)
scaler = GradScaler("cuda")
 
lr_scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)
warmup_sched  = warmup_lr_scheduler(optimizer, warmup_iters=WARMUP_ITERS)
 
print(f"Backbone LR: {BASE_LR/10:.1e}  |  Head LR: {BASE_LR:.1e}")

### LUT Warm-start implementation
To warm-start the LUT, we run a forward pass on the entire training set before the first epoch, storing the initial features for each identity. This gives the model a better starting point for Re-ID learning, as the LUT is populated with meaningful features from the outset rather than starting from zeros.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def warmstart_lut(model, train_loader, device):
    
    """
    Run one eval pass over the training set, average direction embeddings
    per labeled PID at GT boxes, write into the OIM LUT.
    """
    model.eval()
    num_pids = model.model.roi_heads.oim_loss.num_pids
    embed_dim = model.embed_dim

    sum_feats = torch.zeros(num_pids, embed_dim, device=device)
    counts    = torch.zeros(num_pids, device=device)

    for images, targets in tqdm(train_loader, desc="LUT warmstart"):
        images = [img.to(device) for img in images]
        for img, tgt in zip(images, targets):
            pids = tgt["pids"].to(device)
            boxes = tgt["boxes"].to(device)
            labeled_mask = pids >= 0
            if not labeled_mask.any():
                continue
            l_boxes = boxes[labeled_mask]
            l_pids  = pids[labeled_mask]
            for box, pid in zip(l_boxes, l_pids):
                emb = model.extract_query_embedding(img.unsqueeze(0), box).squeeze(0)
                sum_feats[pid] += emb
                counts[pid] += 1

    valid = counts > 0
    avg = sum_feats[valid] / counts[valid].unsqueeze(1)
    avg = F.normalize(avg, dim=1)

    # Replace LUT only for valid pids; leave others as-is (already EMA-updated from epoch 1)
    valid_pids = valid.nonzero(as_tuple=True)[0]
    model.model.roi_heads.oim_loss.lut[valid_pids] = avg

    print(f"LUT warmstarted: {valid.sum().item()}/{num_pids} identities seeded")
    model.train()

### Training function
In this cell we call the `train` function, which handles the entire training loop. For further details on the implementation, please refer to `train.py`.

In [ ]:
# Set to True to train the model from scratch, False to load pretrained weights and skip training
DO_TRAIN = False
 
history = []
 
if DO_TRAIN:
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_losses = train_one_epoch(
            model            = model,
            optimizer        = optimizer,
            data_loader      = train_loader,
            device           = device,
            epoch            = epoch,
            scaler           = scaler,           
            warmup_scheduler = warmup_sched if epoch == 1 else None,
            print_freq       = 200,
            grad_clip        = 10.0,
        )
        lr_scheduler.step()
        history.append(epoch_losses)
 
        print(
            f"Epoch {epoch:02d} summary — "
            f"reid: {epoch_losses.get('loss_reid', 0):.4f}  "
            f"total: {epoch_losses.get('loss_total', 0):.4f}  "
            f"lr: {optimizer.param_groups[1]['lr']:.2e}"
        )
            
        # weights checkpoint
        if epoch >= 16 and epoch % 4 == 0:
            ckpt = f"model_epoch_{epoch:02d}.pt"
            torch.save(model.state_dict(), ckpt)
            print(f'Checkpoint saved: {ckpt}')
 
    torch.save(model.state_dict(), "model_final.pt")
    print("Training complete.")
 
else:
    WEIGHTS = WEIGHTS_PATH  
    model.load_state_dict(torch.load(WEIGHTS, map_location=device))
    model.eval()
    print(f"Loaded weights from {WEIGHTS}")

### Evaluation

After training we extract gallery detections + embeddings and query embeddings, then run the course-provided `eval_search_prw` function (SeqNet-based [Li & Miao, 2021]).

The `det_thresh` parameter — the minimum NAE score required for a gallery detection to be considered — is a sensitive hyperparameter. Too low and the gallery is flooded with false-positive distractors; too high and correctly detected persons are filtered out. We sweep this value at evaluation time (it requires no retraining) and report all results:


In [ ]:
gallery_dets, gallery_feats = build_gallery_dets_feats(model, gallery_loader, device)
query_box_feats = extract_query_feats(model, query_dataset, device)

# Sweep det_thresh values and collect results
sweep = {}
for det_thresh in [0.2, 0.3, 0.4, 0.5, 0.55]:
    ret = eval_search_prw(
        gallery_dataset, query_dataset,
        gallery_dets, gallery_feats, query_box_feats,
        det_thresh=det_thresh,
    )
    sweep[det_thresh] = (ret["mAP"] * 100, float(ret["accs"][0]) * 100)

print()
print(f"{'det_thresh':>11} | {'mAP':>7} | {'Top-1':>7}")
print("-" * 33)
for t, (m, a) in sweep.items():
    marker = "  ← best" if (m, a) == max(sweep.values(), key=lambda x: x[0]) else ""
    print(f"{t:>11.2f} | {m:>6.2f}% | {a:>6.2f}%{marker}")

best_t = max(sweep, key=lambda t: sweep[t][0])
best_map, best_top1 = sweep[best_t]
print()
print(f"Best operating point: det_thresh = {best_t}  →  mAP = {best_map:.2f}%, Top-1 = {best_top1:.2f}%")


### Training loss curves

We plot all three losses — classification, box regression, and Re-ID (OIM) — over training epochs.


In [ ]:
if history:
    epochs = range(1, len(history) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(epochs, [h["loss_total"] for h in history], color="purple")
    ax1.set_title("Total Loss")
    ax1.set_xlabel("Epoch")

    for key, color, label in [
        ("loss_classifier_s1", "steelblue",  "cls s1"),
        ("loss_classifier_s2", "dodgerblue", "cls s2 (NAE)"),
        ("loss_box_reg_s1",    "coral",      "reg s1 (×10)"),
        ("loss_box_reg_s2",    "tomato",     "reg s2"),
        ("loss_reid",          "seagreen",   "reid (OIM)"),
    ]:
        if key in history[0]:
            ax2.plot(epochs, [h[key] for h in history], label=label, color=color)
    ax2.set_title("Individual Losses")
    ax2.set_xlabel("Epoch")
    ax2.legend()

    plt.tight_layout()
    plt.show()

---

## 4. Results and Analysis

This section summarizes the full experimental progression of the project. Each row of the ablation table below corresponds to **a single change applied on top of the previous configuration**, all other settings held fixed. The numbers reported are the best mAP / Top-1 reached after a complete training run and inference-time threshold sweep.


### 4.1 Final result

| Metric | Score |
|---|---|
| **mAP** | **27.21%** |
| **Top-1** | **59.70%** |
| Best `det_thresh` | 0.3 |
| Detection recall (gallery) | 98.99% |

Up from a 20.00% mAP / 51.00% top-1 baseline — a **+7.21 mAP / +8.70 top-1 improvement** through principled, isolated changes.


### 4.2 Ablation summary

| # | Configuration | $\mathcal{L}_{\text{reid}}$ | mAP | Top-1 | $\Delta$ mAP |
|---|---|---|---|---|---|
| 1 | Baseline (Faster R-CNN + OIM, default config) | — | 20.00% | 51.00% | — |
| 2 | + OIM momentum $0.8 \rightarrow 0.5$, input $900/1500$, fg-only ReID head, cosine LR | 2.18 | 23.96% | 59.21% | +3.96 |
| 3 | + Norm-Aware Embedding (NAE) | 1.85 | 24.04% | 59.65% | +0.08 |
| 4 | + Inference-time `det_thresh` calibration | 1.85 | 26.22% | 60.18% | +2.18 |
| 5 | + ResNet-50 FPN v2 backbone | **1.39** | **27.21%** | 59.70% | +0.99 |

A few observations from this table are worth highlighting:

- **The largest single jump came from threshold calibration**, not from an architectural change. After NAE was introduced (Row 3), the *training objective* improved substantially ($\mathcal{L}_{\text{reid}}$ dropped from 2.18 → 1.85), but mAP only moved 0.08. Sweeping `det_thresh` from 0.05 → 0.5 then recovered 2.18 mAP without retraining (Row 4). This is the precision-recall trade-off discussed in Lecture 4 (Slide 12) made concrete: NAE produced a well-calibrated norm-based score, but the default threshold admitted too many low-confidence detections, polluting the matching step.

- **The training-objective vs. downstream-metric divergence widens at the end.** From Row 4 to Row 5, $\mathcal{L}_{\text{reid}}$ dropped a further 25% (1.85 → 1.39) but mAP only moved 1 point. Top-1 actually *decreased* slightly (60.18 → 59.70). This suggests we are approaching a regime where embedding quality is no longer the binding constraint on retrieval performance — see Section 4.4 below.


### 4.3 Negative results

The following configurations were tested during development and **regressed performance**. They are reported here as constraints on the design space — knowing what *doesn't* help is as informative as knowing what does.

| Configuration | $\Delta$ mAP | Likely cause |
|---|---|---|
| Scene-level Random Erasing | $-0.27$ | At full-scene level, erasing rectangles can blot out entire labeled persons, polluting OIM LUT updates. Random Erasing is designed for *person crops*, not scenes. |
| Batch-hard triplet (weight 0.5, batch size 2) | $-0.27$ | With batch size 2 and PRW's typical 1–2 labeled persons per frame, < 5% of training iterations contain a valid triplet. The sparse, high-variance gradient signal does not outweigh the noise added. |
| LUT warmstart after epoch 1 | $-0.36$ | The ReID head's BatchNorm running statistics are not yet stable after 1 epoch. Seeded LUT entries reflect a feature distribution slightly off from the model's eventual one; subsequent training must "unlearn" the mismatch. |
| Soft-NMS at inference | $\approx 0$ | PRW scenes are not crowded enough for Soft-NMS to recover persons hard NMS would suppress. The mechanism doesn't apply when there's nothing being incorrectly suppressed. |


### 4.4 Analysis: training objective vs. downstream metric

Across all five configurations, $\mathcal{L}_{\text{reid}}$ decreased monotonically from 2.18 → 1.39 (a 36% relative improvement on the OIM training objective). The corresponding mAP gain was only 14% relative (23.96% → 27.21%), and Top-1 moved less than 1% relative.

This non-linearity carries a concrete implication: **the model is approaching a metric ceiling determined by factors other than embedding quality.** We can decompose these factors:

1. **Detection geometry.** Even at 99% detection recall, the exact IoU between predicted and ground-truth boxes affects RoI Align quality and therefore embedding consistency. A gallery box with IoU 0.6 to the true person produces a different embedding than the same person at IoU 0.95 — and `eval_search_prw` cannot tell them apart from gallery score alone.

2. **Distractor pedestrian variance.** PRW contains many unlabeled persons whose appearances overlap the labeled-identity manifold. The CQ partially handles them, but at batch size 2 only ~50 distractors per epoch are added against a circular buffer of size 5000.

3. **Inference-time score calibration.** The largest single gain in our ablation (Row 4, +2.18 mAP) came from configuration of the inference pipeline, not training. This generalizes: architectural changes and inference-time configuration must be evaluated *jointly*. A practitioner who only looked at the immediate post-NAE mAP (Row 3, +0.08) would conclude the technique didn't work.

Together, these suggest that further substantial mAP gains on this architecture would come from changing **how matching is done** rather than making embeddings better. Methods like CBGM [Li & Miao, 2021], which post-process the gallery similarity matrix using bipartite matching across scenes, attack the matching procedure directly and consistently add 1–3 mAP on PRW.


### 4.5 Limitations

- **Test-set threshold tuning.** The `det_thresh = 0.3` value was selected by evaluating on the official test split. Strictly speaking this constitutes mild test-set leakage; in a production setting threshold selection should use a held-out validation split. The course's evaluation framework does not provide one, so this is consistent with how prior PRW work in the literature reports.

- **Batch size limitation.** At input resolution 900/1500 with mixed-precision training, batch size 2 was the largest that fit in 16GB of GPU memory. Larger batches with PK-sampling ($P$ identities $\times$ $K$ samples per batch) would unlock useful triplet supervision, but were out of scope for this work's compute budget.

- **No CBGM at inference.** Context-Based Graph Matching is a known +1–3% mAP post-processing step on PRW used by SeqNet and similar works. We did not implement it.
